# Regular Portfolio Optimization Benchmark Tutorial
## Classic 3-Asset Portfolio vs VOO

This notebook demonstrates the **original Qfolio workflow** using a simple 3-asset portfolio (AAPL, MSFT, AMZN) benchmarked against VOO and binary encoding.

### What You'll Learn:
1.  **Data Management**: Fetch and load historical stock data using `DataManager`
2.  **Portfolio Setup**: Configure portfolio and benchmark assets
3.  **Benchmark Simulation**: Run buy-and-hold benchmark for comparison
4.  **Portfolio Optimization**: Run quantum/classical optimization with rebalancing
5.  **Visualization**: Compare optimized portfolio vs individual stocks and VOO
6.  **Assumption**: No transaction cost, rebalance a day after look-back date at adjusted close price

## References 

1.  [Qfolio Paper - GitHub](https://github.com/pcchouCR97/Qfolio/blob/main/paper/paper.pdf)
2.  [Qiskit Finance - IBM](https://github.com/qiskit-community/qiskit-finance/blob/stable/0.4/docs/tutorials/01_portfolio_optimization.ipynb)
3.  [QAOA - Qiskit Algorithms](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.QAOA.html#qiskit_algorithms.QAOA.sampler) 
4.  [SamplerVQE - Qiskit Algorithms](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.SamplingVQE.html) 
5.  [Portfolio Optimization using D-Wave](https://github.com/dwave-examples/portfolio-optimization/tree/main)

## 1. Setup and Imports

In [10]:
import os
import sys
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# Qfolio modules
sys.path.append('..')  # Add parent directory
from qfolio.data.DataManager import DataManager
from qfolio.backtesting.PortfolioManager_AMSP import PortfolioManager

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Modules loaded successfully!")

Modules loaded successfully!


## 2. Mathematical Formulation

### Portfolio Optimization Problem

We solve the mean-variance optimization problem:

Let's first consider using the index $i$ to denote a particular stock, with an average period return per dollar spent of $r_{i}$ for each stock $i$. Let $\sigma_{i,j}$ be the covariance of returns of stocks $i$ and $j$. For a spending budget of $B$ dollars, we can formulate our problem as following:

$$
\begin{aligned}
\text{min} \quad & \sum_{i=1}^{n}\sum_{j=1}^{n} \sigma_{ij}x_{i}x_{j} - \sum_{i=1}^{n} \overline{r_{i}} x_{i}\\
\text{s.t.} \quad &\sum_{i=1}^{n} x_{i} \leq B, \\
            & x_{i} \geq 0, \quad \text{for } i \in \{1, \dots, n\}.
\end{aligned}
$$

**Where:**
-   $x$: is the asset. $x=1$ if selected, $x=0$ if not selected.
-   $\sigma \in \mathbb{R}^{N \times N}$: Covariance matrix of returns
-   $\overline{r_{i}}\in \mathbb{R}^N$: Expected return vector
-   $q$: Risk aversion coefficient (higher = more return-seeking)
-   $\lambda$: Budget constraint penalty (we use $\lambda_1 = 10^3$ in our portfolios example)
-   $B$: Total budget ($)

Next, let's expand our example to let $x_{i}$ denote the number of shares of stock $i$ purchased at price $p_{i}$. We have 

$$
\begin{aligned}
    \text{min} \quad  & \sum_{i=1}^{n}\sum_{j=1}^{n} \sigma_{ij}p_{i}x_{i}p_{j}x_{j} - \sum_{i=1}^{n} \overline{r_{i}} p_{i}x_{i}  \\
    \text{s.t.} \quad  &\sum_{i=1}^{n} p_{i}x_{i} \leq B, \\
                & x_{i} \geq 0, \quad \text{for } i \in \{1, \dots, n\}.
\end{aligned}
$$

### Binary encoding

The binary encoding is the breakthrough of the portfolio optimization compared with the regular approach from [IBM - Qiskit Finance - Portfolio Optimization](https://qiskit-community.github.io/qiskit-finance/tutorials/01_portfolio_optimization.html)

To better reflect real-world investment scenarios, we aim to determine the number of shares $s_i$ for each asset $i$ (with unit price $p_i$), without introducing additional continuous variables, in order to preserve the quadratic form of our optimization problem.

To achieve this, we adopt a binary encoding scheme. Each integer share variable $s_i$ is represented via binary expansion:

$$
s_i = 2^{0}x_{i,0} + 2^{1}x_{i,1} + 2^{2}x_{i,2} + \cdots + 2^{m-1}x_{i,m-1},
$$

where $x_{i,j} \in \{0,1\}$ are binary decision variables, and $m$ is the number of binary bits required to represent the maximum allowable value of $s_i$. For instance, if the maximum number of shares is 5, then $m = 3$ suffices, yielding encodings such as: 0 = 000, 1 = 001, 2 = 010, ..., 5 = 101. This binary formulation enables us to model realistic discrete asset quantities while maintaining compatibility with QUBO formulations.

Using this encoding, we reformulate our original problem as:

$$
\begin{aligned}
\min_{s_1, \dots, s_n} \quad & \sum_{i=1}^{n} \sum_{j=1}^{n} \sigma_{ij} s_i s_j  - \sum_{i=1}^{n} \overline{r_i} s_i\\
\text{s.t.} \quad & \sum_{i=1}^{n} s_i \leq B, \\
& s_i \geq 0, \quad \text{for all } i \in \{1, \dots, n\}.
\end{aligned}
$$

And the corresponding Hamiltonian: 

$$
H = q(\sum_{i=1}^{n}\sum_{j=1}^{n} \sigma_{ij}p_{i}x_{i}p_{j}x_{j}) + \lambda_{1}(\sum_{i=1}p_{i}x_{i} - B)^{2} - \sum_{i=1}^{n}\overline{r_{i}} p_{i}x_{i}
$$

Then, by implementing the binary encoding, we can reformulate above Hamiltonian equation into QUBO form as 

$$
H = q(\sum_{i=1}^{n}\sum_{j=1}^{n} \sigma_{ij}s_{i}s_{j}) + \lambda_{1}(\sum_{i=1}s_{i} - B)^{2} - \sum_{i=1}^{n}\overline{r_{i}} s_{i}
$$

> This converts our problem to **Quadratic Unconstrained Binary Optimization** (QUBO), solvable by QAOA/VQE.

Please see [Qfolio, a quantum-based and classical Numpy-based solver for portfolio optimization package](https://github.com/pcchouCR97/Qfolio/blob/main/paper/paper.pdf) for more mathematical detail explanations!

### Number of bits for base-k encoding (k=2 for binary)
An important implementation detail lies in the selection of $m$, the number of binary bits used per asset. This determines the upper bound on share quantity and is computed as:

$$
m = \left\lceil \log_2\left( \frac{B}{\min(p_i)} \right) \right\rceil, \quad \text{for } i \in \{1, \dots, n\},
$$

where $B$ is the total investment budget, $p_i$ is the price of asset $i$, and $\lceil \cdot \rceil$ denotes the ceiling function. This guarantees that the encoded variable $s_i$ can span from 0 to the maximum number of affordable shares given budget $B$ and asset prices.

In [11]:
# ==================== CONFIGURATION ====================

# Assets
history_tickers = ['AAPL', 'MSFT', 'AMZN', 'VOO']  # Download these tickers
assets_portfolio = ['AAPL', 'MSFT', 'AMZN']  # Optimize over these 3 stocks
assets_benchmark = ['VOO']  # Benchmark against VOO

# Data fetching (set fetch=True to download fresh data)
history_start_date = '2021-02-01'  # Start of data download
history_end_date = '2025-09-06'    # End of data download
fetch_data = False  # Set True to download from Yahoo Finance
save_to_file = False  # Set True to save downloaded data

# Load pre-downloaded data (choose one):
# Option 1: 3-year period (2021-2024)
data_file = "data_VOO_benchmark.csv"

# Backtest period (default: 1 year in 2024)
bm_start_date = "2024-01-01"
bm_end_date = "2024-12-30"

# Portfolio configuration
initial_budget = 10000  # Starting capital ($)
new_invest = 500  # Additional investment per rebalance ($)

# Rebalancing frequencies
freq = '20B'   # Rebalance every 20 business days (~1 month)
tfreq = '20B'   # Training window: 20 business days (~1 month)

# Optimization parameters
k = 2           # Encoding bits per asset
lambda1 = 1E3   # Budget constraint penalty
q = 1E-3        # Risk aversion coefficient
H_scale = 1     # Hamiltonian scaling, default 1 for this case.

# Solver selection (choose one):
solver_type = 'classic'  # Options: 'classic', 'QAOA', 'QAOA_shots', 'SamplerVQE', 'SCIP', 'exact_miqp'

print("Configuration loaded:")
print(f"  Portfolio Assets: {assets_portfolio}")
print(f"  Benchmark: {assets_benchmark}")
print(f"  Backtest Period: {bm_start_date} to {bm_end_date}")
print(f"  Initial Budget: ${initial_budget:,}")
print(f"  New Investment per Rebalance: ${new_invest:,}")
print(f"  Rebalance Frequency: {freq}")
print(f"  Solver: {solver_type}")

Configuration loaded:
  Portfolio Assets: ['AAPL', 'MSFT', 'AMZN']
  Benchmark: ['VOO']
  Backtest Period: 2024-01-01 to 2024-12-30
  Initial Budget: $10,000
  New Investment per Rebalance: $500
  Rebalance Frequency: 20B
  Solver: classic


## 3. Configuration

Modify these parameters to experiment with different settings:

In [12]:
# Initialize DataManager
DM = DataManager(
    tickers=history_tickers,
    start_date=history_start_date,
    end_date=history_end_date
)

if fetch_data:
    print("Fetching data from Yahoo Finance...")
    DM.fetch_yahoo_finance_data(
        price='Close',
        fetch=False,
        save_to_file=save_to_file
    )
    print("✓ Data fetched successfully!")
else:
    print("Skipping data fetch (using pre-downloaded file)")

Skipping data fetch (using pre-downloaded file)


## 4. Data Management

We will use the pre-downloaded file for this jupyter notebook!

In [13]:
# Load data from CSV
print(f"Loading data from: {data_file}")
data = DM.load_data(path=data_file)

print(f"\nData loaded successfully!")
print(f"  Date range: {data.index[0]} to {data.index[-1]}")
print(f"  Total assets: {len(data.columns)}")
print(f"  Total trading days: {len(data)}")
print(f"\nColumns: {data.columns.tolist()}")

# Display first few rows
data.head()

Loading data from: data_VOO_benchmark.csv
Loading data from: data_VOO_benchmark.csv

Data loaded successfully!
  Date range: 2021-10-01 00:00:00 to 2024-12-30 00:00:00
  Total assets: 4
  Total trading days: 816

Columns: ['AAPL', 'AMZN', 'MSFT', 'VOO']


,AAPL,AMZN,MSFT,VOO
Date,,,,
2021-10-01,139.809113,164.162994,280.199829,377.442902
2021-10-04,136.369049,159.488998,274.394257,372.799591
2021-10-05,138.299850,161.050003,279.870270,376.421570
2021-10-06,139.172119,163.100494,284.086334,378.095398
2021-10-07,140.436401,165.121506,285.772858,381.348572


## 5. Initialize Portfolio Manager

In [14]:
PM = PortfolioManager(
    data=data,
    budget=initial_budget,
    new_invest=new_invest,
    assets_portfolio=assets_portfolio,
    assets_benchmark=assets_benchmark
)

print(f"  Portfolio assets: {PM.assets_portfolio}")
print(f"  Benchmark assets: {PM.assets_benchmark}")
print(f"  Initial budget: ${PM.budget:,}")

  Portfolio assets: ['AAPL', 'MSFT', 'AMZN']
  Benchmark assets: ['VOO']
  Initial budget: $10,000


## 6. Run Benchmark Simulation

This returns the results if all budget goes into one of the following asset across the benchmark timeline, VOO, AAPL, MSFT, and AMZN

In [15]:
print(f"Running benchmark simulation from {bm_start_date} to {bm_end_date}...")
print(f"Rebalance frequency: {freq}")
print(f"="*53 + "\n")
print(f"="*15 + "Run benchmark simulator" + "="*15+ "\n")
bm_base = PM.run_benchmark_simulator(
    start=bm_start_date,
    end=bm_end_date,
    save=True,
    save_path="VOO_tutorial_data",
    freq=freq
)

print("="*10 + " Benchmark simulation complete! "+"="*10)

# Display final returns
print("\nFinal Returns (%) in all-in in one asset:")
for ticker in bm_base.columns:
    final_return = bm_base[ticker].iloc[-1]
    print(f"  {ticker}: {final_return:.2f}%")

Running benchmark simulation from 2024-01-01 to 2024-12-30...
Rebalance frequency: 20B

===============Run benchmark simulator===============

Valid business dates for benchmarking:
DatetimeIndex(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05',
               '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11',
               '2024-01-12', '2024-01-16',
               ...
               '2024-12-16', '2024-12-17', '2024-12-18', '2024-12-19',
               '2024-12-20', '2024-12-23', '2024-12-24', '2024-12-26',
               '2024-12-27', '2024-12-30'],
              dtype='datetime64[ns]', name='Date', length=251, freq=None)
Current rebalance points (crps):
DatetimeIndex(['2024-01-02', '2024-01-31', '2024-02-29', '2024-03-28',
               '2024-04-26', '2024-05-24', '2024-06-25', '2024-07-24',
               '2024-08-21', '2024-09-19', '2024-10-17', '2024-11-14',
               '2024-12-13'],
              dtype='datetime64[ns]', name='Date', freq=None)
========== Bench

## 7. Run Portfolio Optimization using Numpy solver

This runs the quantum/classical optimization with periodic rebalancing with `Numpy solver`. Here, the `classic` refers to the ` NumPyMinimumEigensolver()` and so we call it `Numpy solver`. The following is the code snippet from the optimizer and you can have different optimizer options such as `QAOA` and `SamplerVQE`. Note that both quantum algorithms use the default `statevector` method, which we can change to sampling-based for more memory relaxation on local or classical computers. You can also refer to [QAOA - Qiskit Algorithms](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.QAOA.html#qiskit_algorithms.QAOA.sampler) and [SamplerVQE - Qiskit Algorithms](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.SamplingVQE.html) for more info!

For faster results, we set `COBYLA(maxiter=50)` and `reps = 1` for `SamplerVQE` while keeping `COBYLA(maxiter=250)` and `reps = 3` for `QAOA`!

---

```python 
    if self.solver_type == 'classic' or self.solver_type == 'classic_linear_budget':
                classical_algorithm = NumPyMinimumEigensolver()
                classical_eigensolver = MinimumEigenOptimizer(classical_algorithm)
                print(" --- Solving: Classic solver --- ")
                return classical_eigensolver.solve(qp)
                       
            elif self.solver_type == 'QAOA':
                qaoa_optimizer = COBYLA(maxiter=250) # selected so it achieves higher accuracy
                qaoa_sampler = QAOA(sampler=Sampler(), optimizer=qaoa_optimizer, reps=3) # selected so it achieves higher accuracy
                qaoa_res = MinimumEigenOptimizer(qaoa_sampler)
                print(" --- Solving: Quantum solver (QAOA) --- ")
                return qaoa_res.solve(qp)

            elif self.solver_type == 'SamplerVQE':
                vqe_optimzer = COBYLA(maxiter=50)
                vqe_ansatz = TwoLocal(num_assets, "ry", "cz", reps=1, entanglement="full")
                vqe_sampler = SamplingVQE(sampler=Sampler(), ansatz=vqe_ansatz, optimizer=vqe_optimzer)
                vqe_res = MinimumEigenOptimizer(vqe_sampler)
                print(" --- Solving: Quantum solver (SamplerVQE) --- ")
                return vqe_res.solve(qp)



In [16]:
print(f"="*5 + f" Running portfolio optimization with {solver_type} solver" + "="*5)
print(f"  Solver: {solver_type}")
print(f"  Parameters: k={k}, lambda1={lambda1:.0e}, q={q:.0e}")
print(f"  Rebalance freq: {freq}, Training freq: {tfreq}\n")

opt_res = PM.run_portfolio_optimization(
    start=bm_start_date,
    end=bm_end_date,
    save=True,
    save_path="VOO_tutorial_data",
    custom_prefix=f"tutorial_",
    k=k,
    lambda1=lambda1,
    q=q,
    freq=freq,
    t_freq=tfreq,
    H_scale=H_scale,
    solver_type=solver_type
)

print("\n Portfolio optimization complete!")
print(f"  Final Optimized Return: {opt_res.iloc[-1]:.2f}%")

===== Running portfolio optimization with classic solver=====
  Solver: classic
  Parameters: k=2, lambda1=1e+03, q=1e-03
  Rebalance freq: 20B, Training freq: 20B

budget: 10000, price: [251.59307861 423.20291138 221.30000305], k: 2
7
 TRAINING START date: 2023-12-01 00:00:00
 TRAINING END date: 2023-12-29 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-01-02 00:00:00
 (N)ext (R)eturn (P)eroid 2024-01-31 00:00:00
78.74015748031496
Mean Period Returns Vector (r_i):
 AAPL    0.000123
MSFT    0.000047
AMZN    0.000641
Name: 2024-12-30 00:00:00, dtype: float64

Covariance Matrix (sigma):
 [[8.30787000e-06 2.05596165e-06 5.94153609e-06]
 [2.05596165e-06 2.84051002e-06 5.57131344e-06]
 [5.94153609e-06 5.57131344e-06 1.89270175e-05]]
AMSP value: 78.74015748031496
 --- Solving: Classic solver --- 


C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\optimization\hamiltonian.py:143: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  h_3 = - mdl.sum(self._expected_returns[i] * s[i] for i in range(num_assets))


Expected binary solution size: 21
Actual binary solution size: 21 (should be the same as 'Expected binary solution size')
Number of binary bit used: 7

----- Portfolio Share Allocation on 2024-01-02 00:00:00-----
AAPL: 4.694494652693422 shares @ Price 184.2904052734375
MSFT: 20.83846164739673 shares @ Price 366.7073974609375
AMZN: 0.0 shares @ Price 149.92999267578125
8506.76835990527
==== this 2024-01-02 00:00:00 train loop end ====
 TRAINING START date: 2024-01-02 00:00:00
 TRAINING END date: 2024-01-30 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-01-31 00:00:00
 (N)ext (R)eturn (P)eroid 2024-02-29 00:00:00
75.20738421603077
Mean Period Returns Vector (r_i):
 [0.00030805 0.00105402 0.00159207]

Covariance Matrix (sigma):
 [[2.78509320e-05 4.87227793e-06 2.11754848e-05]
 [4.87227793e-06 2.53904424e-06 7.68846374e-06]
 [2.11754848e-05 7.68846374e-06 4.12488461e-05]]
AMSP value: 75.20738421603077
 --- Solving: Classic solver --- 


C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\utils\print_n_plots.py:134: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  share_allocation[asset] = int(share_value) * lots[i]


Expected binary solution size: 21
Actual binary solution size: 21 (should be the same as 'Expected binary solution size')
Number of binary bit used: 7

----- Portfolio Share Allocation on 2024-01-31 00:00:00-----
AAPL: 0.0 shares @ Price 183.05941772460935
MSFT: 26.04620976170337 shares @ Price 393.11761474609375
AMZN: 0.0 shares @ Price 155.1999969482422
10239.223854697251
==== this 2024-01-31 00:00:00 train loop end ====
 TRAINING START date: 2024-01-31 00:00:00
 TRAINING END date: 2024-02-28 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-02-29 00:00:00
 (N)ext (R)eturn (P)eroid 2024-03-28 00:00:00
87.97268211648937
Mean Period Returns Vector (r_i):
 [-0.00036517  0.00033527  0.0034149 ]

Covariance Matrix (sigma):
 [[1.47490912e-05 4.71990828e-06 1.14921296e-05]
 [4.71990828e-06 7.89846243e-06 2.87616687e-05]
 [1.14921296e-05 2.87616687e-05 1.71199504e-04]]
AMSP value: 87.97268211648937
 --- Solving: Classic solver --- 
Expected binary solution size: 21
Actual binary solution size: 21 

C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\backtesting\PortfolioOptimizer_AMSP.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  allocation_results = pd.concat([allocation_results, summary_row])


You can expand the textbox and see the following values for the `classic` solver. You should get all the share allocations and the value of the optimized portfolio, 20384, at the last rebalance period, and the allocation suggested on the date 2024-12-13.

## 8. Performance Comparison Table

Here we will give a quick and simple return comparison table! Note that `PM.run_portfolio_optimization` only returns ROI, not all value across the whole timeframe.

In [17]:
import pandas as pd

# Create comparison table
comparison = pd.DataFrame({
    'Strategy': ['Optimized Portfolio'] + list(bm_base.columns),
    'Final Return (%)': [opt_res.iloc[-1]] + [bm_base[ticker].iloc[-1] for ticker in bm_base.columns]
})

# Calculate outperformance vs VOO
voo_return = bm_base['VOO'].iloc[-1]
comparison['Outperformance vs VOO (%)'] = comparison['Final Return (%)'] - voo_return

# Sort by final return
comparison = comparison.sort_values('Final Return (%)', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print(" " * 20 + "PERFORMANCE COMPARISON")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

# Highlight winner
winner = comparison.iloc[0]['Strategy']
winner_return = comparison.iloc[0]['Final Return (%)']
print(f"\n Best Performer: {winner} ({winner_return:.2f}%)")


                    PERFORMANCE COMPARISON
           Strategy  Final Return (%)  Outperformance vs VOO (%)
               AMZN         37.838288                  18.254143
               AAPL         26.539811                   6.955666
Optimized Portfolio         19.874443                   0.290298
                VOO         19.584145                   0.000000
               MSFT         12.459119                  -7.125026

 Best Performer: AMZN (37.84%)


## 9. QAOA and SamplerVQE

Let's run the following code box to get results from `QAOA` solvers! If any of these solvers run too long on the local machine, we can try:

If it takes too long to run, you can try: 
1.  Longer rebalance periods for faster results
2.  Shorter optimizer iterations for faster convergence
3.  Lower reps for QAOA/SamplerVQE ansatz
4.  Try different classical solvers such as [SPSA (gradient descent) - Qiskit Algorithms](https://qiskit-community.github.io/qiskit-algorithms/stubs/qiskit_algorithms.optimizers.SPSA.html) for optimizing systems with multiple unknown parameters
5.  Try different ansatz, such as [EfficientSU2 - Qiskit](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.EfficientSU2)




In [ ]:
# ~ 9 mins run time on author's local machine
solver_type = 'QAOA'
print(f"="*5 + f" Running portfolio optimization with {solver_type} solver" + "="*5)
print(f"  Solver: {solver_type}")
print(f"  Parameters: k={k}, lambda1={lambda1:.0e}, q={q:.0e}")
print(f"  Rebalance freq: {freq}, Training freq: {tfreq}\n")

opt_res_QAOA = PM.run_portfolio_optimization(
    start=bm_start_date,
    end=bm_end_date,
    save=True,
    save_path="VOO_tutorial_data",
    custom_prefix=f"tutorial_",
    k=k,
    lambda1=lambda1,
    q=q,
    freq=freq,
    t_freq=tfreq,
    H_scale=H_scale,
    solver_type=solver_type
)

print("\n Portfolio optimization complete!")
print(f"  Final Optimized Return: {opt_res_QAOA.iloc[-1]:.2f}%")

===== Running portfolio optimization with classic solver=====
  Solver: classic
  Parameters: k=2, lambda1=1e+03, q=1e-03
  Rebalance freq: 20B, Training freq: 20B

budget: 10000, price: [251.59307861 423.20291138 221.30000305], k: 2
7
 TRAINING START date: 2023-12-01 00:00:00
 TRAINING END date: 2023-12-29 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-01-02 00:00:00
 (N)ext (R)eturn (P)eroid 2024-01-31 00:00:00
78.74015748031496
Mean Period Returns Vector (r_i):
 AAPL    0.000123
MSFT    0.000047
AMZN    0.000641
Name: 2024-12-30 00:00:00, dtype: float64

Covariance Matrix (sigma):
 [[8.30787000e-06 2.05596165e-06 5.94153609e-06]
 [2.05596165e-06 2.84051002e-06 5.57131344e-06]
 [5.94153609e-06 5.57131344e-06 1.89270175e-05]]
AMSP value: 78.74015748031496
 --- Solving: Classic solver --- 


C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\optimization\hamiltonian.py:143: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  h_3 = - mdl.sum(self._expected_returns[i] * s[i] for i in range(num_assets))


Expected binary solution size: 21
Actual binary solution size: 21 (should be the same as 'Expected binary solution size')
Number of binary bit used: 7

----- Portfolio Share Allocation on 2024-01-02 00:00:00-----
AAPL: 4.694494652693422 shares @ Price 184.2904052734375
MSFT: 20.83846164739673 shares @ Price 366.7073974609375
AMZN: 0.0 shares @ Price 149.92999267578125
8506.76835990527
==== this 2024-01-02 00:00:00 train loop end ====
 TRAINING START date: 2024-01-02 00:00:00
 TRAINING END date: 2024-01-30 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-01-31 00:00:00
 (N)ext (R)eturn (P)eroid 2024-02-29 00:00:00
75.20738421603077
Mean Period Returns Vector (r_i):
 [0.00030805 0.00105402 0.00159207]

Covariance Matrix (sigma):
 [[2.78509320e-05 4.87227793e-06 2.11754848e-05]
 [4.87227793e-06 2.53904424e-06 7.68846374e-06]
 [2.11754848e-05 7.68846374e-06 4.12488461e-05]]
AMSP value: 75.20738421603077
 --- Solving: Classic solver --- 


C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\utils\print_n_plots.py:134: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  share_allocation[asset] = int(share_value) * lots[i]


Expected binary solution size: 21
Actual binary solution size: 21 (should be the same as 'Expected binary solution size')
Number of binary bit used: 7

----- Portfolio Share Allocation on 2024-01-31 00:00:00-----
AAPL: 0.0 shares @ Price 183.05941772460935
MSFT: 26.04620976170337 shares @ Price 393.11761474609375
AMZN: 0.0 shares @ Price 155.1999969482422
10239.223854697251
==== this 2024-01-31 00:00:00 train loop end ====
 TRAINING START date: 2024-01-31 00:00:00
 TRAINING END date: 2024-02-28 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-02-29 00:00:00
 (N)ext (R)eturn (P)eroid 2024-03-28 00:00:00
87.97268211648937
Mean Period Returns Vector (r_i):
 [-0.00036517  0.00033527  0.0034149 ]

Covariance Matrix (sigma):
 [[1.47490912e-05 4.71990828e-06 1.14921296e-05]
 [4.71990828e-06 7.89846243e-06 2.87616687e-05]
 [1.14921296e-05 2.87616687e-05 1.71199504e-04]]
AMSP value: 87.97268211648937
 --- Solving: Classic solver --- 
Expected binary solution size: 21
Actual binary solution size: 21 

C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\backtesting\PortfolioOptimizer_AMSP.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  allocation_results = pd.concat([allocation_results, summary_row])


In [ ]:
# ~ 4.5 mins run time on author's local machine
solver_type = 'SamplerVQE'
print(f"="*5 + f" Running portfolio optimization with {solver_type} solver" + "="*5)
print(f"  Solver: {solver_type}")
print(f"  Parameters: k={k}, lambda1={lambda1:.0e}, q={q:.0e}")
print(f"  Rebalance freq: {freq}, Training freq: {tfreq}\n")

opt_res_SamplerVQE = PM.run_portfolio_optimization(
    start=bm_start_date,
    end=bm_end_date,
    save=True,
    save_path="VOO_tutorial_data",
    custom_prefix=f"tutorial_",
    k=k,
    lambda1=lambda1,
    q=q,
    freq=freq,
    t_freq=tfreq,
    H_scale=H_scale,
    solver_type=solver_type
)

print("\n Portfolio optimization complete!")
print(f"  Final Optimized Return: {opt_res_SamplerVQE.iloc[-1]:.2f}%")

===== Running portfolio optimization with SamplerVQE solver=====
  Solver: SamplerVQE
  Parameters: k=2, lambda1=1e+03, q=1e-03
  Rebalance freq: 20B, Training freq: 20B

budget: 10000, price: [251.59307861 423.20291138 221.30000305], k: 2
7
 TRAINING START date: 2023-12-01 00:00:00
 TRAINING END date: 2023-12-29 00:00:00
 (C)urrent (R)eturn (P)eroid: 2024-01-02 00:00:00
 (N)ext (R)eturn (P)eroid 2024-01-31 00:00:00
78.74015748031496
Mean Period Returns Vector (r_i):
 AAPL    0.000123
MSFT    0.000047
AMZN    0.000641
Name: 2024-12-30 00:00:00, dtype: float64

Covariance Matrix (sigma):
 [[8.30787000e-06 2.05596165e-06 5.94153609e-06]
 [2.05596165e-06 2.84051002e-06 5.57131344e-06]
 [5.94153609e-06 5.57131344e-06 1.89270175e-05]]
AMSP value: 78.74015748031496
 --- Solving: Quantum solver (SamplerVQE) --- 


C:\Users\berli\OneDrive\CR_Quantum_Computing\Qfolio_Package\qfolio\optimization\hamiltonian.py:143: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  h_3 = - mdl.sum(self._expected_returns[i] * s[i] for i in range(num_assets))


Solver timed out after 1 minutes.


## 10. Visualization: ROI Comparison

In [ ]:
plt.figure(figsize=(14, 8))

# Plot optimized portfolio
plt.plot(opt_res, label="Optimized Portfolio (Numpy)", linewidth=3, color='green', zorder=10)
plt.plot(opt_res_QAOA, label="Optimized Portfolio (QAOA)", linewidth=3, color='red', zorder=10)
plt.plot(opt_res_SamplerVQE, label="Optimized Portfolio (SamplerVQE)", linewidth=3, color='blue', zorder=10)

# Plot all benchmarks
bm_tickers = assets_portfolio + assets_benchmark

for ticker in bm_tickers:
    if ticker in assets_benchmark:
        # VOO: solid line, thicker
        plt.plot(bm_base[ticker], label=ticker, linewidth=2.5, linestyle="-", color='black')
    else:
        # Individual stocks: dashed lines
        plt.plot(bm_base[ticker], label=ticker, linewidth=1.5, linestyle="--", alpha=0.7)

# Formatting
plt.title(f"Optimized Portfolio vs. Benchmarks ({bm_start_date} to {bm_end_date})\n" +
          f"Solver: {solver_type} | Rebalance: {freq} | Assets: {', '.join(assets_portfolio)}",
          fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=12)
plt.ylabel("Return (%)", fontsize=12)

# Format x-axis dates
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45) # rotate x axis labes by 45 degrees

# Add zero reference line
plt.axhline(0, color='gray', linestyle=':', linewidth=1, alpha=0.5)

# Legend and grid
plt.legend(loc='best', fontsize=11, framealpha=0.9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10.1 Event Analysis: August 2024 Japanese Yen Crisis Response

### Historical Context: The Yen Carry Trade Unwinding

In early August 2024 ("Black Monday"), global financial markets experienced a significant shock due to the unwinding of the Japanese Yen carry trade.

#### SamplerVQE (Blue Line) - EXCEPTIONAL RESILIENCE
-  Pre-crisis (July): Steady climb to ~27%
-  During crisis (Aug 5-10): Minor dip (~3-5%) - significantly outperformed benchmarks
-  Post-crisis recovery: Explosive growth from ~27% → ~45% by year-end
-  Final Return: 45.65% (outperformed VOO by 26.07%)

>  SamplerVQE's `TwoLocal` ansatz with full entanglement likely captured cross-asset correlations better during volatility

#### Numpy Classical (Green Line) - MODERATE IMPACT
-  Pre-crisis (July): Growth to ~21%
-  During crisis: Significant drawdown (~5-7% decline visible)
-  Recovery: Gradual recovery to ~17% by October, final surge to 27.40%
-  Final Return: 27.40% (outperformed VOO by 7.82%)

>  Classical solver was more exposed to the selloff, suggesting heavier allocation to volatile tech stocks (MSFT/AAPL)

#### QAOA (Red Line) - SEVERE UNDERPERFORMANCE
-  Pre-crisis (March-July): Strong early performance (~15%)
-  During crisis: Major decline (dropped to ~6-7%)
-  Recovery attempt: Failed to recover, struggled throughout Q4
-  Final Return: 17.93% (barely outperformed VOO)

>  QAOA's suboptimal performance suggests: improper lambda1 tuning or problem setup or poor ansatz expressivity for capturing risk dynamics.

### Individual Stock Benchmark Response

| Asset | Pre-Crisis | Crisis Drawdown | Recovery | Final Return |
|-----------|---------------|-------------------|-------------|------------------|
| MSFT | ~8% | Worst (-12% to -15%) | Weak recovery | 12.46% |
| AMZN | ~16% | Moderate dip | Explosive Q4 rally | 37.84% |
| AAPL | ~18% | Moderate dip | Steady growth | 26.54% |
| VOO | ~10% | Minor dip | Steady recovery | 19.58%|




## 11. Important Caveats and Parameter Tuning Guidelines

This notebook demonstrates portfolio optimization with predefined quantum solver settings based on heuristic methods. However, careful parameter tuning is essential for achieving optimal results. Below are critical considerations:

### 11.1 Budget Constraint Violations

#### Problem: Under-utilized Capital

If the budget constraint penalty (lambda1) is improperly tuned, the optimizer may not fully invest the available capital:

Example:
```python
# Configuration
initial_budget = 10000  # Expected to invest $10,000
lambda1 = 1E3           # Budget penalty coefficient

# Actual allocation at first rebalance
allocated_value = 9000  # Only $9,000 invested (X)
slack = 1000            # $1,000 uninvested capital
```

This may be due to:
-   lambda1 too weak: Optimizer prioritizes risk minimization over budget utilization
-   lambda1 too strong: Optimizer may violate budget constraint to minimize Hamiltonian


### 11.2 Risk Aversion Coefficient (q) Sensitivity

#### Problem: Extreme Values Lead to Degenerate Solutions

The risk aversion coefficient (q) controls the risk-return tradeoff:

$$
H = q \cdot (\text{risk term}) + \lambda_1 \cdot (\text{budget term}) - (\text{return term})
$$

The choice of `q` can lead to very different results. We propose the following method to adjust our risk aversion coefficient based on the market volatility.

```python
# Estimate current market volatility (VIX proxy)
market_volatility = portfolio_returns.std() * np.sqrt(252)

if market_volatility > 0.25:  # High volatility regime
    q = 1E-2  # Increase risk aversion
elif market_volatility < 0.15:  # Low volatility regime
    q = 1E-4  # Decrease risk aversion (seek returns)
else:
    q = 1E-3  # Baseline
```

### 11.3 Training Window Length (tfreq) Considerations

#### Problem: Overfitting vs Underfitting

The training frequency (`tfreq`) determines how much historical data is used to estimate covariance and expected returns:

Current setting: `tfreq = 20B` (20 business days ≈ 1 month)

Trade-offs:
- Short window (10-20 days): 
  - Pro: Captures recent market regime changes
  - Con: High estimation noise, unstable covariance matrix
  
- Long window (60-252 days):
  - Pro: Stable covariance estimates, lower noise
  - Con: Stale estimates, may miss regime shifts

#### Potential Solution: Exponentially Weighted Moving Average (EWMA)

Instead of uniform weighting, use EMA to balance responsiveness with stability:

```python
from pandas import Series

# Compute EWMA covariance (more weight to recent data)
span = 20  # Effective window
returns = data.pct_change().dropna()
ewma_cov = returns.ewm(span=span).cov()  # Time-varying covariance
```
### 11.4 Summary Checklist for Practitioners

Before deploying a quantum portfolio optimizer in production, verify:

- Budget constraint: proper `lambda_1` setting.
- Parameter sensitivity: proper risk aversion coefficient `q`, dynamic adjustment if needed
- Backtest validation: Test on multiple historical periods (bull, bear, sideways markets)
- Solver comparison: Compare against classical baseline (`Numpy`) and naive benchmarks (equal-weight, buy-and-hold) 

> Always validate on out-of-sample data before risking real capital!